# NOAA AIS 2025 - ĐO LƯỜNG CHẤT LƯỢNG 6 CHIỀU DAMA & KIỂM CHỨNG TIỀN XỬ LÝ

**Đầu vào trực tiếp:** 3 cặp tệp `train.parquet` / `test.parquet` do `DS317_01_Pipeline_Tien_Xu_Ly_Du_Lieu.ipynb` xuất ra:
- `01_ban_tho_18_thuoc_tinh/` (`train.parquet`, `test.parquet`): bản thô 18 cột sau phân chia `TRAIN_PURE` / `TEST_PURE` (đã loại `TRAIN_CROSSING`).
- `02_ban_nen_tang_22_thuoc_tinh/` (`train.parquet`, `test.parquet`): bản nền tảng 22 cột đã khử trùng lặp, lọc `atd_prev`, lọc cự ly nhân quả theo loại tàu (`<= 1,800 NM` Tanker / `<= 2,300 NM` Cargo) và điền khuyết từ `TRAIN_PURE`.
- `03_ban_tien_xu_ly_27_thuoc_tinh/` (`train.parquet`, `test.parquet`): bản 27 cột ML-Ready (kèm `cog_sin/cos`, `heading_sin/cos`, `imputation_flag`).
- Tệp phụ trợ tùy chọn (nếu gắn kèm): `noaa_ais_2025_trip_split_manifest.parquet`, `train_imputation_artifacts.json`.

## Quy tắc kiểm toán & chống rò rỉ
1. Phân chia theo thời gian (`2025-10-01`): `TRAIN_PURE` (`ata_time < 2025-10-01`) vs `TEST_PURE` (`min_voyage_time >= 2025-10-01`), loại bỏ hoàn toàn `TRAIN_CROSSING`.
2. Ngưỡng không gian nhân quả `v_avg * 7 ngày`: **1,800 NM** (Tanker) và **2,300 NM** (Cargo).
3. Lọc bất đối xứng: **Train** lọc `Target_Hours > 0` và `implied_spd >= 1.62 knots`; **Test không lọc theo `Target_Hours`** (giữ cả các mẫu `> 168h`).
4. Mọi thực nghiệm chọn đặc trưng (B2) và hàm mất mát (B3) chỉ sử dụng dữ liệu `Train` (tách nội bộ Jan-Jun / Jul-Sep), không chạm tập `Test Q4` để điều chỉnh mô hình.

In [ ]:
from collections import Counter
import gc
import importlib.util
import json
import os
from pathlib import Path
import platform
import time
import zipfile

from IPython.display import FileLink, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REQUIRED_PACKAGES = ["pyarrow", "lightgbm"]
MISSING_PACKAGES = [
    pkg for pkg in REQUIRED_PACKAGES
    if importlib.util.find_spec(pkg) is None
]
if MISSING_PACKAGES:
    raise ImportError(
        "Thieu package: "
        + ", ".join(MISSING_PACKAGES)
        + ". Hay cai dat truoc khi chay lai."
    )

import lightgbm as lgb
import pyarrow.parquet as pq

SPLIT_CUTOFF_STR = "2025-10-01"
MAX_DIST_TANKER_NM = 1800.0
MAX_DIST_CARGO_NM = 2300.0
DIST_TOLERANCE_NM = 1e-3
MIN_IMPLIED_SPEED_KNOTS = 1.62

OUT = Path(os.getenv("DS317_BT2_OUT", "/kaggle/working/bt2_dama_results"))
BATCH_SIZE = int(os.getenv("DS317_BATCH_SIZE", "500000"))
SAMPLE_TRAIN_SIZE = int(os.getenv("DS317_SAMPLE_TRAIN", "240000"))
SAMPLE_TEST_SIZE = int(os.getenv("DS317_SAMPLE_TEST", "80000"))
SEED = 42
MAX_BATCHES_ENV = os.getenv("DS317_MAX_BATCHES")
MAX_BATCHES = int(MAX_BATCHES_ENV) if MAX_BATCHES_ENV else None
REUSE_SCAN_CACHE = False
KNOWN_DESTINATION_AT_PREDICTION = True
RUN_ASYMMETRIC_EXPERIMENT = True
RUN_EXACT_DUPLICATE_AUDIT = True
DUPLICATE_MEMORY_LIMIT = "4GB"
DUPLICATE_DISK_LIMIT = "8GB"
N_ESTIMATORS = 800
N_JOBS = 4

RAW2 = OUT / "RAW2"
DAMA = OUT / "DAMA"
TASK03 = OUT / "TASK03"
COMMON = OUT / "COMMON"
FIG_DIR = OUT / "Hình ảnh biểu đồ"
for folder in [OUT, RAW2, DAMA, TASK03, COMMON, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print(
    f"Python: {platform.python_version()} | LightGBM: {lgb.__version__} "
    f"| MAX_BATCHES={MAX_BATCHES}"
)

In [ ]:
VERSION_DIRS = (
    "01_ban_tho_18_thuoc_tinh",
    "02_ban_nen_tang_22_thuoc_tinh",
    "03_ban_tien_xu_ly_27_thuoc_tinh",
)


def _has_split_pair(base_dir: Path, sub: str) -> bool:
    """Check whether both train and test parquets exist in subdir."""
    return (
        (base_dir / sub / "train.parquet").is_file()
        and (base_dir / sub / "test.parquet").is_file()
    )


def locate_split_root() -> Path:
    """Locate output directory produced by DS317_01 notebook."""
    env_root = os.getenv("DS317_SPLIT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/kaggle/working"),
        Path("./kaggle_work_fullyear"),
        Path("."),
    ])
    for cand in candidates:
        if cand.exists() and all(
            _has_split_pair(cand, sub) for sub in VERSION_DIRS
        ):
            return cand.resolve()

    kaggle_in = Path("/kaggle/input")
    if kaggle_in.exists():
        for probe in kaggle_in.rglob("02_ban_nen_tang_22_thuoc_tinh"):
            root = probe.parent
            if all(_has_split_pair(root, sub) for sub in VERSION_DIRS):
                return root.resolve()

    found_parquets = (
        [str(p) for p in list(kaggle_in.rglob("*.parquet"))[:15]]
        if kaggle_in.exists()
        else []
    )
    raise FileNotFoundError(
        "Khong tim thay du 3 thu muc train/test parquet cua DS317_01. "
        "Hay gan dataset output vao Kaggle hoac set DS317_SPLIT_ROOT. "
        f"Parquet thay trong /kaggle/input: {found_parquets}"
    )


def find_optional_file(root: Path, filename: str):
    """Find optional manifest or artifacts file near split root."""
    direct = root / filename
    if direct.is_file():
        return direct
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.exists():
        matches = list(kaggle_in.rglob(filename))
        if matches:
            return matches[0]
    return None


def distance_wgs84_np(lon1, lat1, lon2, lat2):
    """Compute WGS84 ellipsoid geodesic distance in km."""
    a = 6378.1370
    b = a * (1.0 - 1.0 / 298.257223563)
    lon1_f = np.asarray(lon1, dtype=np.float64)
    lat1_f = np.asarray(lat1, dtype=np.float64)
    lon2_f = np.asarray(lon2, dtype=np.float64)
    lat2_f = np.asarray(lat2, dtype=np.float64)
    lat_mid = np.radians((lat1_f + lat2_f) * 0.5)
    cos_m = np.cos(lat_mid)
    sin_m = np.sin(lat_mid)
    radius = np.sqrt(
        ((a * a * cos_m) ** 2 + (b * b * sin_m) ** 2)
        / ((a * cos_m) ** 2 + (b * sin_m) ** 2)
    )
    dlat = np.radians(lat2_f - lat1_f)
    dlon = (
        (np.radians(lon2_f - lon1_f) + np.pi) % (2.0 * np.pi)
    ) - np.pi
    h_val = (
        np.sin(dlat * 0.5) ** 2
        + np.cos(np.radians(lat1_f))
        * np.cos(np.radians(lat2_f))
        * (np.sin(dlon * 0.5) ** 2)
    )
    c_val = 2.0 * np.arcsin(np.sqrt(np.clip(h_val, 0.0, 1.0)))
    return (radius * c_val).astype(np.float32)


def save_table(df: pd.DataFrame, path: Path) -> None:
    """Save DataFrame to UTF-8 BOM CSV for Excel compatibility."""
    df.to_csv(path, index=False, encoding="utf-8-sig")


def save_fig(fig, path: Path) -> None:
    """Save Matplotlib figure neatly and close it."""
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)


class StreamingStatTracker:
    """Welford streaming tracker for count, mean, std, min, max."""

    def __init__(self, total_expected: int = 0) -> None:
        self.total = int(total_expected)
        self.missing = 0
        self.n_valid = 0
        self.mean = 0.0
        self.m2 = 0.0
        self.min_val = np.inf
        self.max_val = -np.inf

    def update(self, series: pd.Series) -> None:
        """Update tracker with a numeric or datetime batch Series."""
        arr = pd.to_numeric(series, errors="coerce").to_numpy(
            dtype=np.float64
        )
        valid = np.isfinite(arr)
        self.missing += int((~valid).sum())
        vals = arr[valid]
        b_n = len(vals)
        if b_n == 0:
            return
        b_mean = float(vals.mean())
        b_m2 = float(((vals - b_mean) ** 2).sum())
        if self.n_valid == 0:
            self.mean = b_mean
            self.m2 = b_m2
            self.n_valid = b_n
        else:
            delta = b_mean - self.mean
            new_n = self.n_valid + b_n
            self.m2 += b_m2 + (delta * delta * self.n_valid * b_n / new_n)
            self.mean += delta * b_n / new_n
            self.n_valid = new_n
        self.min_val = min(self.min_val, float(vals.min()))
        self.max_val = max(self.max_val, float(vals.max()))

    def summary(self) -> dict:
        """Return exact summary statistics dictionary."""
        std = (
            float(np.sqrt(self.m2 / (self.n_valid - 1)))
            if self.n_valid > 1
            else np.nan
        )
        return {
            "Count_Valid": self.n_valid,
            "Missing": self.missing,
            "Min": float(self.min_val) if self.n_valid else np.nan,
            "Mean": float(self.mean) if self.n_valid else np.nan,
            "Max": float(self.max_val) if self.n_valid else np.nan,
            "Std": std,
        }


class StreamingCorrOLS:
    """Exact streaming Pearson correlation and OLS (y = slope*x + b)."""

    def __init__(self) -> None:
        self.n = 0
        self.mx = 0.0
        self.my = 0.0
        self.xx = 0.0
        self.yy = 0.0
        self.xy = 0.0

    def update(self, x_arr, y_arr) -> None:
        """Accumulate pairwise finite observations."""
        x = np.asarray(x_arr, dtype=np.float64)
        y = np.asarray(y_arr, dtype=np.float64)
        ok = np.isfinite(x) & np.isfinite(y)
        x = x[ok]
        y = y[ok]
        b_n = len(x)
        if b_n == 0:
            return
        bx = float(x.mean())
        by = float(y.mean())
        dx = bx - self.mx
        dy = by - self.my
        new_n = self.n + b_n
        factor = self.n * b_n / new_n if self.n > 0 else 0.0
        self.xx += float(((x - bx) ** 2).sum()) + dx * dx * factor
        self.yy += float(((y - by) ** 2).sum()) + dy * dy * factor
        self.xy += float(((x - bx) * (y - by)).sum()) + dx * dy * factor
        self.mx += dx * b_n / new_n
        self.my += dy * b_n / new_n
        self.n = new_n

    def result(self) -> dict:
        """Return Pearson r, OLS slope, and OLS intercept."""
        if self.n < 2 or self.xx <= 0.0 or self.yy <= 0.0:
            return {"n": self.n, "r": np.nan, "slope": np.nan, "bias": np.nan}
        r_val = self.xy / np.sqrt(self.xx * self.yy)
        slope = self.xy / self.xx
        bias = self.my - slope * self.mx
        return {
            "n": self.n,
            "r": float(r_val),
            "slope": float(slope),
            "bias": float(bias),
        }


SPLIT_ROOT = locate_split_root()
MANIFEST_PATH = find_optional_file(
    SPLIT_ROOT, "noaa_ais_2025_trip_split_manifest.parquet"
)
ARTIFACTS_JSON_PATH = find_optional_file(
    SPLIT_ROOT, "train_imputation_artifacts.json"
)
PATHS = {
    sub: {
        "train": SPLIT_ROOT / sub / "train.parquet",
        "test": SPLIT_ROOT / sub / "test.parquet",
    }
    for sub in VERSION_DIRS
}
print("Split Root:", SPLIT_ROOT)
for sub, pair in PATHS.items():
    print(f"  {sub}: train={pair['train'].name} | test={pair['test'].name}")
print("Optional manifest :", MANIFEST_PATH)
print("Optional artifacts:", ARTIFACTS_JSON_PATH)

In [ ]:
inv_rows = []
for ver_name, pair in PATHS.items():
    for split_name in ("train", "test"):
        p = pair[split_name]
        pf_meta = pq.ParquetFile(p)
        size_bytes = p.stat().st_size
        codec = "unknown"
        if pf_meta.num_row_groups > 0:
            codec = str(pf_meta.metadata.row_group(0).column(0).compression)
        inv_rows.append({
            "version": ver_name,
            "split": split_name,
            "rows": int(pf_meta.metadata.num_rows),
            "columns": int(pf_meta.metadata.num_columns),
            "row_groups": int(pf_meta.num_row_groups),
            "compression": codec,
            "size_mb": round(size_bytes / (1024 ** 2), 2),
            "size_gib": round(size_bytes / (1024 ** 3), 4),
        })

df_inventory = pd.DataFrame(inv_rows)
save_table(df_inventory, COMMON / "dataset_versions_inventory.csv")
display(df_inventory)

if MANIFEST_PATH is not None:
    df_man = pd.read_parquet(MANIFEST_PATH)
    man_sum = (
        df_man.groupby("split_category", as_index=False)
        .agg(
            trips=("trip_id", "nunique"),
            unique_mmsi=("mmsi", "nunique"),
            raw_voyage_rows=("num_raw_voyage_rows", "sum"),
        )
    )
    save_table(man_sum, COMMON / "split_manifest_summary.csv")
    display(man_sum)
    del df_man

# A. KIỂM KÊ BẢN THÔ 18 CỘT & KIỂM TOÁN DAMA 6 CHIỀU BẢN 22 CỘT
## A1. Quét toàn bộ `01_ban_tho_18_thuoc_tinh` (`train.parquet` + `test.parquet`)

Đo lường trạng thái thô trước khi lọc hàng hải, khử trùng lặp và điền khuyết (nhưng đã tách sẵn `TRAIN_PURE` / `TEST_PURE`, loại bỏ `TRAIN_CROSSING`). Lấy mẫu đều tỷ lệ trên cả Train và Test để phục vụ kiểm chứng bộ lọc A3 và thử nghiệm điền khuyết A4.

In [ ]:
COLS_18 = pq.read_schema(
    PATHS["01_ban_tho_18_thuoc_tinh"]["train"]
).names
STAT_COLS_18 = ["Target_Hours", "sog", "cog", "draft", "length", "width"]


def scan_raw18_split(path: Path, split_label: str, sample_target: int):
    """Stream one 18-col parquet split and collect audit + sample."""
    pf_obj = pq.ParquetFile(path)
    total_rows = pf_obj.metadata.num_rows
    rng = np.random.default_rng(
        SEED + (0 if split_label == "Train" else 1)
    )
    counts = Counter()
    vessels = Counter()
    statuses = Counter()
    trackers = {c: StreamingStatTracker(total_rows) for c in STAT_COLS_18}
    null_counts = {c: 0 for c in COLS_18}

    if MAX_BATCHES is None:
        positions = np.sort(
            rng.choice(
                total_rows,
                size=min(sample_target, total_rows),
                replace=False,
            )
        )
    else:
        est_rows = min(total_rows, MAX_BATCHES * BATCH_SIZE)
        positions = np.sort(
            rng.choice(
                est_rows,
                size=min(sample_target, est_rows),
                replace=False,
            )
        )

    sog_edges = np.linspace(0.0, 110.0, 221)
    target_edges = np.linspace(0.0, 336.0, 169)
    hist_sog = np.zeros(len(sog_edges) - 1, dtype=np.int64)
    hist_target = np.zeros(len(target_edges) - 1, dtype=np.int64)
    parts = []
    offset = 0

    for b_idx, batch in enumerate(
        pf_obj.iter_batches(batch_size=BATCH_SIZE, columns=COLS_18)
    ):
        if MAX_BATCHES is not None and b_idx >= MAX_BATCHES:
            break
        df_b = batch.to_pandas()
        n_rows = len(df_b)
        counts["N_rows"] += n_rows

        for c in COLS_18:
            null_counts[c] += int(df_b[c].isna().sum())
        for c in STAT_COLS_18:
            trackers[c].update(df_b[c])

        sog_v = pd.to_numeric(df_b["sog"], errors="coerce").to_numpy(float)
        fs = np.isfinite(sog_v)
        tgt_v = pd.to_numeric(
            df_b["Target_Hours"], errors="coerce"
        ).to_numpy(float)
        fy = np.isfinite(tgt_v)

        masks = {
            "SOG_nonfinite": ~fs,
            "SOG_zero": fs & (sog_v == 0.0),
            "SOG_1_to_40": fs & (sog_v >= 1.0) & (sog_v <= 40.0),
            "SOG_over_40": fs & (sog_v > 40.0),
            "Target_nonfinite": ~fy,
            "Target_le_0": fy & (tgt_v <= 0.0),
            "Target_0_to_168": fy & (tgt_v > 0.0) & (tgt_v <= 168.0),
            "Target_over_168": fy & (tgt_v > 168.0),
        }
        for key, mask in masks.items():
            counts[key] += int(mask.sum())

        hist_sog += np.histogram(sog_v[fs], bins=sog_edges)[0]
        hist_target += np.histogram(
            tgt_v[fy & (tgt_v >= 0.0) & (tgt_v <= 336.0)],
            bins=target_edges,
        )[0]

        for c, counter in [("vessel_type", vessels), ("status", statuses)]:
            v = pd.to_numeric(df_b[c], errors="coerce")
            labels = v.where(np.isfinite(v)).astype("string").fillna("Missing")
            counter.update(labels.value_counts().to_dict())

        lo = np.searchsorted(positions, offset)
        hi = np.searchsorted(positions, offset + n_rows)
        if hi > lo:
            sub = df_b.iloc[positions[lo:hi] - offset].copy()
            sub["split"] = split_label
            parts.append(sub)
        offset += n_rows
        del df_b, batch

    df_sample = (
        pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    )
    return {
        "counts": counts,
        "trackers": trackers,
        "nulls": null_counts,
        "vessels": vessels,
        "statuses": statuses,
        "hist_sog": hist_sog,
        "hist_target": hist_target,
        "sog_edges": sog_edges,
        "target_edges": target_edges,
        "sample": df_sample,
        "scanned": offset,
    }


raw18_tr = scan_raw18_split(
    PATHS["01_ban_tho_18_thuoc_tinh"]["train"], "Train", SAMPLE_TRAIN_SIZE
)
raw18_te = scan_raw18_split(
    PATHS["01_ban_tho_18_thuoc_tinh"]["test"], "Test", SAMPLE_TEST_SIZE
)

sample_raw18 = pd.concat(
    [raw18_tr["sample"], raw18_te["sample"]], ignore_index=True
)
total_raw18_scanned = raw18_tr["scanned"] + raw18_te["scanned"]

raw18_audit_rows = []
for metric in sorted(
    set(raw18_tr["counts"]) | set(raw18_te["counts"])
):
    c_tr = int(raw18_tr["counts"][metric])
    c_te = int(raw18_te["counts"][metric])
    c_all = c_tr + c_te
    raw18_audit_rows.append({
        "metric": metric,
        "train_count": c_tr,
        "train_pct": round(100.0 * c_tr / max(raw18_tr["scanned"], 1), 4),
        "test_count": c_te,
        "test_pct": round(100.0 * c_te / max(raw18_te["scanned"], 1), 4),
        "total_count": c_all,
        "total_pct": round(100.0 * c_all / max(total_raw18_scanned, 1), 4),
    })

raw18_audit_df = pd.DataFrame(raw18_audit_rows)
save_table(raw18_audit_df, RAW2 / "raw18_audit.csv")
display(raw18_audit_df)

raw18_null_df = pd.DataFrame([
    {
        "column": c,
        "train_nulls": raw18_tr["nulls"][c],
        "train_null_pct": round(
            100.0 * raw18_tr["nulls"][c] / max(raw18_tr["scanned"], 1), 4
        ),
        "test_nulls": raw18_te["nulls"][c],
        "test_null_pct": round(
            100.0 * raw18_te["nulls"][c] / max(raw18_te["scanned"], 1), 4
        ),
        "total_null_pct": round(
            100.0
            * (raw18_tr["nulls"][c] + raw18_te["nulls"][c])
            / max(total_raw18_scanned, 1),
            4,
        ),
    }
    for c in COLS_18
])
save_table(raw18_null_df, RAW2 / "raw18_missing_pre_imputation.csv")
display(raw18_null_df)

In [ ]:
hist_sog_all = raw18_tr["hist_sog"] + raw18_te["hist_sog"]
sog_edges = raw18_tr["sog_edges"]
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax in axs:
    ax.stairs(hist_sog_all, sog_edges, fill=True, color="#2878B5")
    ax.set_xlabel("SOG (knots)")
axs[0].set_xlim(0, 40)
axs[0].set_ylabel("So ban ghi")
axs[0].set_title("Ban tho 18 cot - SOG 0-40 kn")
axs[1].set_yscale("log")
axs[1].axvline(40, color="red", ls="--")
axs[1].set_title("Phan duoi SOG (truc log)")
save_fig(fig, RAW2 / "01_sog.png")

for key_name, fname, title in [
    ("vessels", "02_vessel", "Phan bo loai tau (Ban 18 cot)"),
    ("statuses", "03_status", "Trang thai AIS (Ban 18 cot)"),
]:
    comb = raw18_tr[key_name] + raw18_te[key_name]
    df_tab = pd.DataFrame(
        list(comb.items()), columns=["value", "count"]
    ).sort_values("count", ascending=False)
    df_tab["pct"] = 100.0 * df_tab["count"] / max(total_raw18_scanned, 1)
    save_table(df_tab, RAW2 / f"{fname}.csv")
    fig, ax = plt.subplots(figsize=(11, 4))
    df_tab.head(20).set_index("value")["pct"].plot.bar(
        ax=ax, color="#4C72B0", rot=45
    )
    ax.set_ylabel("% Ban ghi 18 cot")
    ax.set_title(title)
    save_fig(fig, RAW2 / f"{fname}.png")

## A2. Kiểm toán DAMA 6 Chiều & Từ điển 22 Thuộc tính trên `02_ban_nen_tang_22_thuoc_tinh`

Đo đạc trực tiếp trên cả `train.parquet` (Q1-Q3) và `test.parquet` (Q4) của Bản 22 cột:
- **Completeness:** Kiểm chứng 21/22 cột đạt 0% khuyết sau điền khuyết từ `TRAIN_PURE`, đồng thời đo tỷ lệ `cargo` giữ `NaN` gốc.
- **Consistency (Quy tắc mới):** C1 kiểm tra `Dist_To_Berth_NM <= 1,800 NM` (Tanker) hoặc `<= 2,300 NM` (Cargo) thay cho ngưỡng cũ `Target_Hours <= 168h`; ghi nhận riêng hành vi không lọc `Target_Hours` ở tập Test.
- **Accuracy:** Đo vận tốc biểu kiến `>= 1.62 knots`, Pearson `r(Dist_To_Berth_NM, Target_Hours)` và hồi quy tuyến tính OLS trên toàn bộ dữ liệu.
- **Timeliness:** Kiểm tra phạm vi tiếp cận 7 ngày (`Dist_To_Berth_NM <= v_avg * 7d`, năm 2025, `t < ata_time`), độ phủ 365/365 ngày và phân bố Q1-Q4 / 12 tháng.
- **Validity & Uniqueness:** Kiểm định miền giá trị chuẩn ITU/IMO/WGS84 trên 22 cột và kiểm tra trùng lặp `(mmsi, base_date_time)` bằng DuckDB (nếu khả dụng) trên cả Bản 18 cột lẫn Bản 22 cột.

In [ ]:
COLS_22 = pq.read_schema(
    PATHS["02_ban_nen_tang_22_thuoc_tinh"]["train"]
).names
STAT_22_COLS = [
    c for c in COLS_22 if c not in ("trip_id", "base_date_time", "ata_time")
]


def _quarter_from_dt(dt_series: pd.Series) -> pd.Series:
    """Assign Q1-Q4 string labels cleanly from timestamp Series."""
    q_num = dt_series.dt.quarter
    return "Q" + q_num.astype("Int64").astype(str)


def audit_base22_full():
    """Stream all batches of 02_ban_nen_tang_22 train and test."""
    tr_path = PATHS["02_ban_nen_tang_22_thuoc_tinh"]["train"]
    te_path = PATHS["02_ban_nen_tang_22_thuoc_tinh"]["test"]
    pf_tr = pq.ParquetFile(tr_path)
    pf_te = pq.ParquetFile(te_path)
    n_tr_meta = pf_tr.metadata.num_rows
    n_te_meta = pf_te.metadata.num_rows

    rng_tr = np.random.default_rng(SEED + 10)
    rng_te = np.random.default_rng(SEED + 11)
    lim_tr = (
        n_tr_meta if MAX_BATCHES is None
        else min(n_tr_meta, MAX_BATCHES * BATCH_SIZE)
    )
    lim_te = (
        n_te_meta if MAX_BATCHES is None
        else min(n_te_meta, MAX_BATCHES * BATCH_SIZE)
    )
    pos_tr = np.sort(
        rng_tr.choice(
            lim_tr, size=min(SAMPLE_TRAIN_SIZE, lim_tr), replace=False
        )
    )
    pos_te = np.sort(
        rng_te.choice(
            lim_te, size=min(SAMPLE_TEST_SIZE, lim_te), replace=False
        )
    )

    col_trackers = {c: StreamingStatTracker() for c in STAT_22_COLS}
    col_nulls_tr = {c: 0 for c in COLS_22}
    col_nulls_te = {c: 0 for c in COLS_22}
    corr_tr = StreamingCorrOLS()
    corr_te = StreamingCorrOLS()
    corr_all = StreamingCorrOLS()

    ships = {"Train": set(), "Test": set(), "All": set()}
    trips = {"Train": set(), "Test": set()}
    dates_covered = set()
    monthly_counts = Counter()

    q_buckets = {
        q: Counter() for q in ("Q1", "Q2", "Q3", "Q4", "Train", "Test", "All")
    }
    sample_parts = []
    cols_non_cargo = [c for c in COLS_22 if c != "cargo"]

    for split_name, pf_obj, positions in [
        ("Train", pf_tr, pos_tr),
        ("Test", pf_te, pos_te),
    ]:
        offset = 0
        for b_idx, batch in enumerate(
            pf_obj.iter_batches(batch_size=BATCH_SIZE, columns=COLS_22)
        ):
            if MAX_BATCHES is not None and b_idx >= MAX_BATCHES:
                break
            df_b = batch.to_pandas()
            n_b = len(df_b)

            dt_b = pd.to_datetime(
                df_b["base_date_time"], errors="coerce", utc=True
            )
            ata_b = pd.to_datetime(
                df_b["ata_time"], errors="coerce", utc=True
            )
            q_lbl = _quarter_from_dt(dt_b)
            dates_covered.update(dt_b.dt.date.dropna().unique().tolist())
            m_vc = dt_b.dt.strftime("%Y-%m").dropna().value_counts()
            monthly_counts.update(m_vc.to_dict())

            ships[split_name].update(df_b["mmsi"].dropna().unique().tolist())
            ships["All"].update(df_b["mmsi"].dropna().unique().tolist())
            trips[split_name].update(
                df_b["trip_id"].dropna().unique().tolist()
            )

            null_map = col_nulls_tr if split_name == "Train" else col_nulls_te
            for c in COLS_22:
                null_map[c] += int(df_b[c].isna().sum())
            for c in STAT_22_COLS:
                col_trackers[c].update(df_b[c])

            is_tk = df_b["Is_Tanker"].to_numpy() == 1
            dist_nm = df_b["Dist_To_Berth_NM"].to_numpy(dtype=np.float64)
            sog_v = df_b["sog"].to_numpy(dtype=np.float64)
            tgt_v = df_b["Target_Hours"].to_numpy(dtype=np.float64)
            st_v = df_b["status"].to_numpy(dtype=np.float64)

            if split_name == "Train":
                corr_tr.update(dist_nm, tgt_v)
            else:
                corr_te.update(dist_nm, tgt_v)
            corr_all.update(dist_nm, tgt_v)

            max_dist = np.where(
                is_tk, MAX_DIST_TANKER_NM, MAX_DIST_CARGO_NM
            )
            c1 = np.isfinite(dist_nm) & (dist_nm >= 0.0) & (
                dist_nm <= max_dist + DIST_TOLERANCE_NM
            )
            c2 = np.isfinite(sog_v) & (sog_v >= 1.0) & (sog_v <= 40.0)
            c3 = np.isfinite(st_v) & (st_v != 1.0) & (st_v != 5.0)
            c4 = (dt_b < ata_b).to_numpy()
            cons_ok = c1 & c2 & c3 & c4

            imp_spd = np.where(
                np.isfinite(tgt_v) & (tgt_v > 0.0), dist_nm / tgt_v, -1.0
            )
            acc_ok = np.isfinite(dist_nm) & (imp_spd >= MIN_IMPLIED_SPEED_KNOTS)
            time_ok = (dt_b.dt.year == 2025).to_numpy() & c1 & c4
            tgt_gt_168 = np.isfinite(tgt_v) & (tgt_v > 168.0)
            tgt_le_0 = np.isfinite(tgt_v) & (tgt_v <= 0.0)

            val_ok = (
                df_b["mmsi"].between(201_000_000, 775_999_999).to_numpy()
                & df_b["latitude"].between(15.0, 75.0).to_numpy()
                & df_b["longitude"].between(-180.0, -65.0).to_numpy()
                & df_b["vessel_type"].between(70, 89).to_numpy()
                & df_b["Is_Tanker"].isin([0, 1]).to_numpy()
                & c2
                & df_b["length"].gt(0.0).to_numpy()
                & df_b["width"].gt(0.0).to_numpy()
                & df_b["draft"].gt(0.0).to_numpy()
                & df_b["draft"].le(25.5).to_numpy()
                & df_b["cog"].ge(0.0).to_numpy()
                & df_b["cog"].lt(360.0).to_numpy()
                & df_b["heading"].ge(0.0).to_numpy()
                & df_b["heading"].lt(360.0).to_numpy()
            )
            comp_rec_all = df_b[COLS_22].notna().all(axis=1).to_numpy()
            comp_rec_nocar = (
                df_b[cols_non_cargo].notna().all(axis=1).to_numpy()
            )

            flags = {
                "n_rows": np.ones(n_b, dtype=bool),
                "tanker_rows": is_tk,
                "comp_rec_all": comp_rec_all,
                "comp_rec_no_cargo": comp_rec_nocar,
                "cons_c1_dist": c1,
                "cons_ok": cons_ok,
                "acc_vapp_ge1_62": acc_ok,
                "timely_ok": time_ok,
                "val_ok": val_ok,
                "target_gt_168": tgt_gt_168,
                "target_le_0": tgt_le_0,
            }

            for q_val in ("Q1", "Q2", "Q3", "Q4"):
                qm = (q_lbl == q_val).to_numpy()
                if not qm.any():
                    continue
                for fk, fmask in flags.items():
                    cnt = int((qm & fmask).sum())
                    q_buckets[q_val][fk] += cnt

            for fk, fmask in flags.items():
                cnt = int(fmask.sum())
                q_buckets[split_name][fk] += cnt
                q_buckets["All"][fk] += cnt

            lo = np.searchsorted(positions, offset)
            hi = np.searchsorted(positions, offset + n_b)
            if hi > lo:
                sub = df_b.iloc[positions[lo:hi] - offset].copy()
                sub["split"] = split_name
                sample_parts.append(sub)
            offset += n_b
            del df_b, batch

    overlap = trips["Train"] & trips["Test"]
    assert len(overlap) == 0, (
        f"Phat hien trip ro ri Train/Test: {len(overlap)}"
    )

    sample_22 = pd.concat(sample_parts, ignore_index=True)
    return {
        "buckets": q_buckets,
        "col_trackers": col_trackers,
        "nulls_tr": col_nulls_tr,
        "nulls_te": col_nulls_te,
        "corr_tr": corr_tr.result(),
        "corr_te": corr_te.result(),
        "corr_all": corr_all.result(),
        "ships": {k: len(v) for k, v in ships.items()},
        "trips": {k: len(v) for k, v in trips.items()},
        "overlap_trips": len(overlap),
        "days_covered": len(dates_covered),
        "monthly_counts": monthly_counts,
        "sample_22": sample_22,
    }


t_dama0 = time.time()
audit22 = audit_base22_full()
print(f"Hoan tat quet DAMA Ban 22 cot trong {time.time() - t_dama0:.1f}s")
print(
    f"Tau (MMSI): Train={audit22['ships']['Train']:,} | "
    f"Test={audit22['ships']['Test']:,} | All={audit22['ships']['All']:,}"
)
print(
    f"Chuyen (trip_id): Train={audit22['trips']['Train']:,} | "
    f"Test={audit22['trips']['Test']:,} | "
    f"Overlap={audit22['overlap_trips']}"
)
print("Pearson/OLS All :", audit22["corr_all"])
print("Pearson/OLS Train:", audit22["corr_tr"])
print("Pearson/OLS Test :", audit22["corr_te"])
print("So ngay bao phu 2025:", audit22["days_covered"])

In [ ]:
dup_rows = []
for ver_name in (
    "01_ban_tho_18_thuoc_tinh",
    "02_ban_nen_tang_22_thuoc_tinh",
):
    for split_name in ("train", "test"):
        p = PATHS[ver_name][split_name]
        rec = {
            "version": ver_name,
            "split": split_name,
            "status": "NOT_RUN",
            "total_rows": None,
            "distinct_keys": None,
            "duplicate_excess": None,
            "uniqueness_pct": None,
        }
        if (
            RUN_EXACT_DUPLICATE_AUDIT
            and MAX_BATCHES is None
            and importlib.util.find_spec("duckdb") is not None
        ):
            import duckdb

            tmp_duck = OUT / "duckdb_tmp"
            tmp_duck.mkdir(exist_ok=True)
            con = duckdb.connect()
            try:
                con.execute(f"SET memory_limit='{DUPLICATE_MEMORY_LIMIT}'")
                con.execute(
                    f"SET max_temp_directory_size='{DUPLICATE_DISK_LIMIT}'"
                )
                tmp_str = str(tmp_duck).replace("'", "''")
                con.execute(f"SET temp_directory='{tmp_str}'")
                con.execute("SET threads=4")
                q_sql = (
                    "SELECT SUM(n-1), COUNT(*), SUM(n) FROM ("
                    "SELECT mmsi, base_date_time, COUNT(*) AS n "
                    "FROM read_parquet(?) "
                    "WHERE mmsi IS NOT NULL AND base_date_time IS NOT NULL "
                    "GROUP BY mmsi, base_date_time) AS sub"
                )
                res = con.execute(q_sql, [str(p)]).fetchone()
                dup_ex = int(res[0] or 0)
                dist_k = int(res[1] or 0)
                tot_k = int(res[2] or 0)
                rec.update({
                    "status": "MEASURED",
                    "total_rows": tot_k,
                    "distinct_keys": dist_k,
                    "duplicate_excess": dup_ex,
                    "uniqueness_pct": (
                        round(100.0 * (1.0 - dup_ex / tot_k), 6)
                        if tot_k
                        else np.nan
                    ),
                })
            except Exception as exc:
                rec["status"] = f"FAILED: {exc}"
            finally:
                con.close()
        dup_rows.append(rec)

df_dup = pd.DataFrame(dup_rows)
save_table(df_dup, DAMA / "uniqueness_duckdb_audit.csv")
display(df_dup)

In [ ]:
bks = audit22["buckets"]
n_tr_22 = bks["Train"]["n_rows"]
n_te_22 = bks["Test"]["n_rows"]
n_all_22 = bks["All"]["n_rows"]

field_comp_all = 100.0 * (
    1.0
    - sum(
        audit22["nulls_tr"][c] + audit22["nulls_te"][c] for c in COLS_22
    )
    / max(n_all_22 * len(COLS_22), 1)
)
cols_no_cargo = [c for c in COLS_22 if c != "cargo"]
field_comp_no_cargo = 100.0 * (
    1.0
    - sum(
        audit22["nulls_tr"][c] + audit22["nulls_te"][c]
        for c in cols_no_cargo
    )
    / max(n_all_22 * len(cols_no_cargo), 1)
)

dama_q_rows = []
for period in ("Q1", "Q2", "Q3", "Train", "Q4", "Test", "All"):
    st = bks[period]
    n_p = max(int(st["n_rows"]), 1)
    dama_q_rows.append({
        "Period": period,
        "N_rows": int(st["n_rows"]),
        "Pct_total": round(100.0 * st["n_rows"] / max(n_all_22, 1), 2),
        "Completeness_Rec_All_pct": round(
            100.0 * st["comp_rec_all"] / n_p, 4
        ),
        "Completeness_Rec_NoCargo_pct": round(
            100.0 * st["comp_rec_no_cargo"] / n_p, 4
        ),
        "Consistency_Spatial7d_pct": round(100.0 * st["cons_ok"] / n_p, 4),
        "Accuracy_Vapp_ge1_62_pct": round(100.0 * st["acc_vapp_ge1_62"] / n_p, 4),
        "Timeliness_Spatial7d_pct": round(100.0 * st["timely_ok"] / n_p, 4),
        "Validity_Domain22_pct": round(100.0 * st["val_ok"] / n_p, 4),
        "Target_gt_168h_pct": round(100.0 * st["target_gt_168"] / n_p, 4),
        "Target_le_0h_pct": round(100.0 * st["target_le_0"] / n_p, 4),
    })

df_dama_q = pd.DataFrame(dama_q_rows)
save_table(df_dama_q, DAMA / "dama_6_dimensions_by_quarter.csv")
display(df_dama_q)

rec_nc_pct = 100.0 * bks["All"]["comp_rec_no_cargo"] / max(n_all_22, 1)
summary_dama = pd.DataFrame([
    {
        "Dimension": "1. Completeness (Record All 22 / NoCargo)",
        "Train_pct": round(
            100.0 * bks["Train"]["comp_rec_all"] / max(n_tr_22, 1), 4
        ),
        "Test_pct": round(
            100.0 * bks["Test"]["comp_rec_all"] / max(n_te_22, 1), 4
        ),
        "All_pct": round(
            100.0 * bks["All"]["comp_rec_all"] / max(n_all_22, 1), 4
        ),
        "Extra_Note": (
            f"Rec_NoCargo={rec_nc_pct:.4f}% | "
            f"Field22={field_comp_all:.4f}% | "
            f"Field21_NoCargo={field_comp_no_cargo:.4f}%"
        ),
    },
    {
        "Dimension": "2. Consistency (C1 Dist<=1800/2300, C2..C4)",
        "Train_pct": round(
            100.0 * bks["Train"]["cons_ok"] / max(n_tr_22, 1), 4
        ),
        "Test_pct": round(100.0 * bks["Test"]["cons_ok"] / max(n_te_22, 1), 4),
        "All_pct": round(100.0 * bks["All"]["cons_ok"] / max(n_all_22, 1), 4),
        "Extra_Note": (
            f"Target>168h: Train={bks['Train']['target_gt_168']:,}, "
            f"Test={bks['Test']['target_gt_168']:,}"
        ),
    },
    {
        "Dimension": "3. Accuracy (Vapp>=1.62 kn & Pearson/OLS)",
        "Train_pct": round(
            100.0 * bks["Train"]["acc_vapp_ge1_62"] / max(n_tr_22, 1), 4
        ),
        "Test_pct": round(
            100.0 * bks["Test"]["acc_vapp_ge1_62"] / max(n_te_22, 1), 4
        ),
        "All_pct": round(
            100.0 * bks["All"]["acc_vapp_ge1_62"] / max(n_all_22, 1), 4
        ),
        "Extra_Note": (
            f"r_all={audit22['corr_all']['r']:.4f} | "
            f"r_train={audit22['corr_tr']['r']:.4f} | "
            f"r_test={audit22['corr_te']['r']:.4f}"
        ),
    },
    {
        "Dimension": "4. Timeliness (Spatial 7d & 2025 Coverage)",
        "Train_pct": round(
            100.0 * bks["Train"]["timely_ok"] / max(n_tr_22, 1), 4
        ),
        "Test_pct": round(
            100.0 * bks["Test"]["timely_ok"] / max(n_te_22, 1), 4
        ),
        "All_pct": round(
            100.0 * bks["All"]["timely_ok"] / max(n_all_22, 1), 4
        ),
        "Extra_Note": f"Days_2025={audit22['days_covered']}/365",
    },
    {
        "Dimension": "5. Validity (Domain 22 cols)",
        "Train_pct": round(
            100.0 * bks["Train"]["val_ok"] / max(n_tr_22, 1), 4
        ),
        "Test_pct": round(100.0 * bks["Test"]["val_ok"] / max(n_te_22, 1), 4),
        "All_pct": round(100.0 * bks["All"]["val_ok"] / max(n_all_22, 1), 4),
        "Extra_Note": "ITU-R M.585 / M.1371 / IMO 70-89 / US BBox",
    },
])
save_table(summary_dama, DAMA / "dama_6_dimensions_summary.csv")
display(summary_dama)

s22 = audit22["sample_22"]
dict_rows = []
for col in COLS_22:
    n_null = audit22["nulls_tr"][col] + audit22["nulls_te"][col]
    row = {
        "Column": col,
        "Count_NonNull": n_all_22 - n_null,
        "Missing": n_null,
        "Missing_Pct": round(100.0 * n_null / max(n_all_22, 1), 4),
    }
    if col in audit22["col_trackers"]:
        st_ex = audit22["col_trackers"][col].summary()
        vals = pd.to_numeric(s22[col], errors="coerce").dropna()
        q25 = float(vals.quantile(0.25)) if len(vals) else np.nan
        med = float(vals.median()) if len(vals) else np.nan
        q75 = float(vals.quantile(0.75)) if len(vals) else np.nan
        row.update({
            "Min": st_ex["Min"],
            "Q25_est": q25,
            "Median_est": med,
            "Mean": st_ex["Mean"],
            "Q75_est": q75,
            "Max": st_ex["Max"],
            "Std": st_ex["Std"],
            "IQR_est": (q75 - q25) if np.isfinite(q25) else np.nan,
        })
    dict_rows.append(row)

df_dict22 = pd.DataFrame(dict_rows)
save_table(df_dict22, DAMA / "data_dictionary_22cols_stats.csv")
display(df_dict22)

df_monthly = pd.DataFrame(
    sorted(audit22["monthly_counts"].items()), columns=["Month", "Rows"]
)
save_table(df_monthly, DAMA / "monthly_counts_2025.csv")
display(df_monthly)

In [ ]:
s22 = audit22["sample_22"]

if len(s22) > 0:
    # Hinh 1: Phan phoi Target_Hours (Train vs Test + duoi > 168h)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    for lbl, color in [("Train", "#2878B5"), ("Test", "#D95319")]:
        sub_y = s22.loc[s22["split"] == lbl, "Target_Hours"].dropna()
        sub_y_pos = sub_y[sub_y >= 0.0]
        if len(sub_y_pos) == 0:
            continue
        axs[0].hist(
            np.log1p(sub_y_pos), bins=60, alpha=0.55, density=True,
            label=lbl, color=color,
        )
        axs[1].hist(
            sub_y_pos.clip(upper=350.0), bins=70, alpha=0.55, density=True,
            label=lbl, color=color,
        )
    axs[0].set_xlabel("log(1 + Target_Hours)")
    axs[0].set_ylabel("Mat do")
    axs[0].set_title(
        "Phan phoi Target_Hours theo thang log (Train vs Test)"
    )
    axs[0].legend()
    axs[1].axvline(168.0, color="black", ls="--", lw=1.2, label="Moc 168h")
    axs[1].set_xlabel("Target_Hours (cat hien thi tai 350h)")
    axs[1].set_title("Phan phoi goc (Test giu ca mau > 168h)")
    axs[1].legend()
    save_fig(fig, FIG_DIR / "Hình 1 - Phân phối Target Hours.png")

    # Hinh 2: Scatter Dist_To_Berth_NM vs Target_Hours (1800/2300 NM)
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    pts = s22.sample(n=min(40_000, len(s22)), random_state=SEED)
    for grp_v, grp_lbl, grp_col in [
        (0, "Cargo", "#1F77B4"),
        (1, "Tanker", "#D95319"),
    ]:
        sub_p = pts.loc[pts["Is_Tanker"] == grp_v]
        if len(sub_p) > 0:
            sub_p.plot.scatter(
                x="Dist_To_Berth_NM",
                y="Target_Hours",
                color=grp_col,
                label=grp_lbl,
                alpha=0.25,
                s=8,
                ax=ax,
            )
    x_line = np.linspace(0.0, MAX_DIST_CARGO_NM, 100)
    ols = audit22["corr_all"]
    if np.isfinite(ols["slope"]):
        ax.plot(
            x_line,
            ols["slope"] * x_line + ols["bias"],
            color="black",
            lw=2,
            label=(
                f"OLS: y={ols['slope']:.4f}x+{ols['bias']:.2f} "
                f"(r={ols['r']:.4f})"
            ),
        )
    ax.axvline(
        MAX_DIST_TANKER_NM, color="#D95319", ls="--", label="Tanker 1,800 NM"
    )
    ax.axvline(
        MAX_DIST_CARGO_NM, color="#1F77B4", ls="--", label="Cargo 2,300 NM"
    )
    ax.axhline(168.0, color="gray", ls=":", label="168h tham chieu")
    ax.set_title("Tuong quan Cu ly WGS84 vs Target_Hours (Ban 22 cot)")
    ax.legend()
    save_fig(fig, FIG_DIR / "Hình 2 - Phân tán Dist vs Target.png")

    # Hinh 3: Boxplot SOG & Draft (Cargo vs Tanker)
    s22_box = s22[["Is_Tanker", "sog", "draft"]].copy()
    s22_box["Vessel_Group"] = s22_box["Is_Tanker"].map(
        {0: "Cargo", 1: "Tanker"}
    )
    s22_box = s22_box.dropna(subset=["Vessel_Group"])
    if len(s22_box) > 0:
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
        s22_box.boxplot(
            column="sog", by="Vessel_Group", ax=axs[0], showfliers=False
        )
        axs[0].set_title("Phan phoi SOG (knots)")
        axs[0].set_xlabel("")
        s22_box.boxplot(
            column="draft", by="Vessel_Group", ax=axs[1], showfliers=False
        )
        axs[1].set_title("Phan phoi Draft (met)")
        axs[1].set_xlabel("")
        fig.suptitle(
            "So sanh Dong hoc & Mon nuoc: Cargo vs Tanker (Ban 22 cot)"
        )
        save_fig(
            fig, FIG_DIR / "Hình 3 - Biểu đồ hộp SOG Draft Cargo vs Tanker.png"
        )

    # Hinh 4: Mat do dia khong gian AIS
    fig, ax = plt.subplots(figsize=(10, 5.5))
    hb = ax.hexbin(
        pts["longitude"],
        pts["latitude"],
        gridsize=85,
        bins="log",
        cmap="YlGnBu",
    )
    fig.colorbar(hb, ax=ax, label="log10(so tin AIS)")
    ax.set_xlabel("Kinh do (Longitude)")
    ax.set_ylabel("Vi do (Latitude)")
    ax.set_title(
        "Mat do khong gian luong tau NOAA AIS 2025 (Ban 22 cot)"
    )
    save_fig(fig, FIG_DIR / "Hình 4 - Mật độ địa không gian.png")

# Bieu do DAMA Q1-Q4 (Hinh 5.1 Heatmap & Hinh 5.2 Bar)
dama_q_only = df_dama_q[
    df_dama_q["Period"].isin(["Q1", "Q2", "Q3", "Q4", "All"])
]
score_cols = [
    "Completeness_Rec_NoCargo_pct",
    "Consistency_Spatial7d_pct",
    "Accuracy_Vapp_ge1_62_pct",
    "Timeliness_Spatial7d_pct",
    "Validity_Domain22_pct",
]
mat = dama_q_only.set_index("Period")[score_cols].T
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(mat.to_numpy(), vmin=80, vmax=100, cmap="RdYlGn")
ax.set_xticks(range(len(mat.columns)))
ax.set_xticklabels(mat.columns)
ax.set_yticks(range(len(mat.index)))
ax.set_yticklabels(mat.index)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(
            j,
            i,
            f"{mat.iloc[i, j]:.2f}%",
            ha="center",
            va="center",
            fontsize=9,
        )
fig.colorbar(im, ax=ax)
ax.set_title("Ma tran Chat luong 6 Chieu DAMA qua Q1-Q4 (Ban 22 cot)")
save_fig(fig, FIG_DIR / "dama_heatmap_q1_q4.png")

fig, ax = plt.subplots(figsize=(10, 4.5))
dama_q_only.set_index("Period")["N_rows"].plot.bar(
    ax=ax, color=["#2878B5", "#2878B5", "#2878B5", "#D95319", "#555555"], rot=0
)
ax.set_ylabel("So ban ghi sach")
ax.set_title("Quy mo ban ghi Ban 22 cot qua Q1-Q4 & All")
save_fig(fig, FIG_DIR / "dama_bar_q1_q4.png")

## A3 & A4. Kiểm chứng Bộ lọc Bất đối xứng (`1,800 / 2,300 NM`) và Điền khuyết 3 cấp (`mmsi -> vessel_type -> Cargo/Tanker -> global`)

- **A3:** Chạy waterfall lọc trên mẫu `01_ban_tho_18_thuoc_tinh` tách riêng `Train` (có lọc `Target_Hours > 0` và `implied_spd >= 1.62 knots`) và `Test` (KHÔNG lọc theo `Target_Hours`), kèm `atd_prev` nếu có `noaa_ais_2025_trip_split_manifest.parquet`.
- **A4:** Fit lookup điền khuyết **đúng 3 cấp** như `DS317_01` (`mmsi -> vessel_type -> Cargo/Tanker -> global` + `status_mode`) trên nửa đầu Train (`Jan-Jun`) và kiểm chứng mask-and-recover 20% trên nửa sau Train (`Jul-Sep`), không dùng Q4 Test.

In [ ]:
atd_prev_map = None
if MANIFEST_PATH is not None:
    man_cols = set(pq.read_schema(MANIFEST_PATH).names)
    if {"trip_id", "atd_prev"} <= man_cols:
        df_m = pd.read_parquet(
            MANIFEST_PATH, columns=["trip_id", "atd_prev"]
        )
        atd_prev_map = dict(zip(df_m["trip_id"], df_m["atd_prev"]))
        del df_m


def enrich_and_filter_raw18(df: pd.DataFrame, is_train: bool):
    """Apply DS317_01 maritime filters and track waterfall counts."""
    d = df.copy()
    for c in (
        "longitude", "latitude", "berth_lon", "berth_lat", "sog",
        "cog", "heading", "status", "vessel_type", "length", "width",
        "draft", "cargo", "Target_Hours",
    ):
        d[c] = pd.to_numeric(d[c], errors="coerce").astype(float)
    d["base_date_time"] = pd.to_datetime(
        d["base_date_time"], errors="coerce", utc=True
    )
    d["ata_time"] = pd.to_datetime(d["ata_time"], errors="coerce", utc=True)

    d["Dist_To_Berth_NM"] = np.nan
    coord_ok = (
        d["latitude"].between(-90.0, 90.0)
        & d["longitude"].between(-180.0, 180.0)
        & d["berth_lat"].between(-90.0, 90.0)
        & d["berth_lon"].between(-180.0, 180.0)
    )
    if coord_ok.any():
        p = d.loc[coord_ok]
        d.loc[coord_ok, "Dist_To_Berth_NM"] = (
            distance_wgs84_np(
                p["longitude"], p["latitude"], p["berth_lon"], p["berth_lat"]
            ) / np.float32(1.852)
        )
    d["Is_Tanker"] = np.where(
        d["vessel_type"].between(70.0, 89.0),
        (d["vessel_type"] >= 80.0).astype(float),
        np.nan,
    )
    max_d = np.where(
        d["Is_Tanker"] == 1.0, MAX_DIST_TANKER_NM, MAX_DIST_CARGO_NM
    )
    rules = [
        ("SOG [1, 40] kn", d["sog"].between(1.0, 40.0)),
        ("Status hop le (~isin {1, 5})", ~d["status"].isin([1.0, 5.0])),
        (
            "Vessel type nguyen [70, 89]",
            d["vessel_type"].between(70.0, 89.0)
            & d["vessel_type"].mod(1).eq(0.0),
        ),
        (
            "US BBox (lat 15-75, lon -180..-65)",
            d["latitude"].between(15.0, 75.0)
            & d["longitude"].between(-180.0, -65.0),
        ),
        ("base_date_time < ata_time", d["base_date_time"] < d["ata_time"]),
        (
            "Cu ly <= 1800 (Tanker) / 2300 (Cargo)",
            d["Dist_To_Berth_NM"].notna() & (d["Dist_To_Berth_NM"] <= max_d),
        ),
    ]
    if atd_prev_map is not None:
        atd_p = pd.to_datetime(
            d["trip_id"].map(atd_prev_map), errors="coerce", utc=True
        )
        rules.append((
            "Sau atd_prev (chuyen truoc)",
            atd_p.isna() | (d["base_date_time"] > atd_p),
        ))
    if is_train:
        y = d["Target_Hours"]
        v_app = d["Dist_To_Berth_NM"].div(y.where(y > 0.0))
        rules.extend([
            ("Train: Target_Hours > 0", y.gt(0.0)),
            ("Train: implied_spd >= 1.62 knots", v_app.ge(MIN_IMPLIED_SPEED_KNOTS)),
        ])

    keep = pd.Series(True, index=d.index)
    wf = [{"step": "Mau raw 18 cot", "remaining": len(d), "removed": 0}]
    for step_name, mask in rules:
        before = int(keep.sum())
        keep &= mask.fillna(False)
        after = int(keep.sum())
        wf.append({
            "step": step_name,
            "remaining": after,
            "removed": before - after,
        })
    return d.loc[keep].copy(), pd.DataFrame(wf)


work_tr_18, wf_tr = enrich_and_filter_raw18(
    sample_raw18.loc[sample_raw18["split"] == "Train"], is_train=True
)
work_te_18, wf_te = enrich_and_filter_raw18(
    sample_raw18.loc[sample_raw18["split"] == "Test"], is_train=False
)
save_table(wf_tr, RAW2 / "sample_filter_waterfall_train.csv")
save_table(wf_te, RAW2 / "sample_filter_waterfall_test.csv")
print("--- Waterfall Mau TRAIN (18 -> loc 22) ---")
display(wf_tr)
print("--- Waterfall Mau TEST (KHONG loc Target_Hours) ---")
display(wf_te)

In [ ]:
BODY_COLS = ["length", "width", "draft"]
LEVELS_3 = [
    ("mmsi", ["mmsi"]),
    ("vessel_type", ["vessel_type"]),
    ("group", ["_ship_group"]),
]
JAN = pd.Timestamp("2025-01-01", tz="UTC")
JUL = pd.Timestamp("2025-07-01", tz="UTC")
OCT = pd.Timestamp("2025-10-01", tz="UTC")


def prepare_body(df: pd.DataFrame) -> pd.DataFrame:
    """Clean invalid body bounds and compute cargo/tanker group."""
    d = df.copy()
    for c in BODY_COLS:
        vals = pd.to_numeric(d[c], errors="coerce")
        ok = np.isfinite(vals) & vals.gt(0.0)
        if c == "draft":
            ok &= vals.le(25.5)
        d[c] = vals.where(ok)
    d["_ship_group"] = pd.Series(pd.NA, index=d.index, dtype="string")
    d.loc[d["vessel_type"].between(70, 79), "_ship_group"] = "Cargo"
    d.loc[d["vessel_type"].between(80, 89), "_ship_group"] = "Tanker"
    return d


def fit_3level_imputer(ref: pd.DataFrame) -> dict:
    """Fit DS317_01 3-level median hierarchy and status mode."""
    r = prepare_body(ref)
    st_valid = pd.to_numeric(r["status"], errors="coerce").dropna()
    st_mode = float(st_valid.mode().iloc[0]) if len(st_valid) else 0.0
    body_lookup = {
        c: {
            "global": float(r[c].median()),
            "maps": [
                (name, keys[0], r.groupby(keys[0], dropna=True)[c].median())
                for name, keys in LEVELS_3
            ],
        }
        for c in BODY_COLS
    }
    return {"body": body_lookup, "status_mode": st_mode}


def transform_3level_imputer(data: pd.DataFrame, lookup: dict) -> pd.DataFrame:
    """Apply DS317_01 3-level imputer plus status/cog/heading fills."""
    d = prepare_body(data)
    for c in BODY_COLS:
        src = pd.Series("observed", index=d.index)
        src.loc[d[c].isna()] = "unresolved"
        for name, key_col, table in lookup["body"][c]["maps"]:
            fill = d[key_col].map(table)
            take = d[c].isna() & fill.notna()
            d.loc[take, c] = fill.loc[take]
            src.loc[take] = name
        take = d[c].isna()
        if pd.notna(lookup["body"][c]["global"]):
            d.loc[take, c] = lookup["body"][c]["global"]
            src.loc[take] = "global"
        d[c + "_source"] = src
    d["status"] = pd.to_numeric(d["status"], errors="coerce").fillna(
        lookup["status_mode"]
    )
    for ang_col in ("cog", "heading"):
        d[ang_col] = pd.to_numeric(d[ang_col], errors="coerce").fillna(0.0)
    d["Area_Proxy"] = d["length"] * d["width"]
    d["Volume_Proxy"] = d["Area_Proxy"] * d["draft"]
    return d


imp_tr = work_tr_18.loc[
    work_tr_18["base_date_time"].ge(JAN)
    & work_tr_18["base_date_time"].lt(JUL)
    & work_tr_18["ata_time"].lt(JUL)
].copy()
imp_va = work_tr_18.loc[
    work_tr_18["base_date_time"].ge(JUL)
    & work_tr_18["base_date_time"].lt(OCT)
    & work_tr_18["ata_time"].lt(OCT)
].copy()

if len(imp_tr) >= 100 and len(imp_va) >= 100:
    overlap_tr_va = set(imp_tr["trip_id"]) & set(imp_va["trip_id"])
    imp_tr = imp_tr.loc[~imp_tr["trip_id"].isin(overlap_tr_va)].copy()
    lu = fit_3level_imputer(imp_tr)
    truth = prepare_body(imp_va)
    masked = truth.copy()
    rng_m = np.random.default_rng(SEED + 21)
    masks = {}
    for c in BODY_COLS:
        elig = truth.index[truth[c].notna()].to_numpy()
        masks[c] = (
            rng_m.choice(
                elig, size=max(1, int(0.2 * len(elig))), replace=False
            )
            if len(elig)
            else np.array([], dtype=int)
        )
        masked.loc[masks[c], c] = np.nan
    rec = transform_3level_imputer(masked, lu)
    m_rows = []
    s_rows = []
    for c in BODY_COLS:
        idx = masks[c]
        act = truth.loc[idx, c].to_numpy(float)
        prd = rec.loc[idx, c].to_numpy(float)
        ok = np.isfinite(act) & np.isfinite(prd)
        err = prd[ok] - act[ok]
        m_rows.append({
            "variable": c,
            "N_masked": len(idx),
            "N_recovered": int(ok.sum()),
            "coverage_pct": 100.0 * ok.mean() if len(ok) else np.nan,
            "MAE": float(np.mean(np.abs(err))) if len(err) else np.nan,
            "RMSE": float(np.sqrt(np.mean(err ** 2))) if len(err) else np.nan,
            "global_median_MAE": (
                float(np.mean(np.abs(act - lu["body"][c]["global"])))
                if len(act)
                else np.nan
            ),
        })
        for nm, cnt in rec.loc[idx, c + "_source"].value_counts().items():
            s_rows.append({"variable": c, "source": nm, "count": int(cnt)})
    df_mask = pd.DataFrame(m_rows)
    df_src = pd.DataFrame(s_rows)
    save_table(df_mask, RAW2 / "table_6_1_mask_recover.csv")
    save_table(df_src, RAW2 / "mask_recover_sources.csv")
    display(df_mask)
    display(df_src)
else:
    print("Canh bao: Mau Jan-Jun/Jul-Sep qua nho de chay A4 mask-recover.")

# B. PHÂN TÍCH BẢN 27 CỘT ML-READY (`03_ban_tien_xu_ly_27_thuoc_tinh`)

Đọc trực tiếp cặp `train.parquet` / `test.parquet` của Bản 27 cột:
- Thống kê `imputation_flag` (`0, 1, 2, 3`) và kiểm chứng 0% missing trên các biến đầu vào ML.
- Tách mẫu `Train` thành `Jan-Jun` (fit) và `Jul-Sep` (valid, Zero Trip Overlap) để thử nghiệm **Hybrid Feature Selection** (Pearson + LightGBM Gain, không PCA) và **Asymmetric Loss**; hoàn toàn không chạm tập `Test Q4`.

In [ ]:
COLS_27 = pq.read_schema(
    PATHS["03_ban_tien_xu_ly_27_thuoc_tinh"]["train"]
).names


def scan_prep27(path: Path, split_label: str, sample_target: int):
    """Scan 27-col parquet for flag counts and uniform sample."""
    pf_obj = pq.ParquetFile(path)
    total_rows = pf_obj.metadata.num_rows
    lim = (
        total_rows if MAX_BATCHES is None
        else min(total_rows, MAX_BATCHES * BATCH_SIZE)
    )
    rng = np.random.default_rng(SEED + (30 if split_label == "Train" else 31))
    positions = np.sort(
        rng.choice(lim, size=min(sample_target, lim), replace=False)
    )
    flag_counts = Counter()
    parts = []
    offset = 0
    for b_idx, batch in enumerate(
        pf_obj.iter_batches(batch_size=BATCH_SIZE, columns=COLS_27)
    ):
        if MAX_BATCHES is not None and b_idx >= MAX_BATCHES:
            break
        df_b = batch.to_pandas()
        flag_counts.update(df_b["imputation_flag"].value_counts().to_dict())
        lo = np.searchsorted(positions, offset)
        hi = np.searchsorted(positions, offset + len(df_b))
        if hi > lo:
            sub = df_b.iloc[positions[lo:hi] - offset].copy()
            sub["split"] = split_label
            parts.append(sub)
        offset += len(df_b)
        del df_b, batch
    return (
        pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(),
        flag_counts,
        offset,
    )


s27_tr, flags_tr, n27_tr = scan_prep27(
    PATHS["03_ban_tien_xu_ly_27_thuoc_tinh"]["train"],
    "Train",
    SAMPLE_TRAIN_SIZE,
)
s27_te, flags_te, n27_te = scan_prep27(
    PATHS["03_ban_tien_xu_ly_27_thuoc_tinh"]["test"],
    "Test",
    SAMPLE_TEST_SIZE,
)

flag_rows = []
for fl in sorted(set(flags_tr) | set(flags_te)):
    c_tr = int(flags_tr.get(fl, 0))
    c_te = int(flags_te.get(fl, 0))
    flag_rows.append({
        "imputation_flag": int(fl),
        "train_count": c_tr,
        "train_pct": round(100.0 * c_tr / max(n27_tr, 1), 4),
        "test_count": c_te,
        "test_pct": round(100.0 * c_te / max(n27_te, 1), 4),
    })
df_flags27 = pd.DataFrame(flag_rows)
save_table(df_flags27, TASK03 / "imputation_flag_distribution_27.csv")
display(df_flags27)

DIST_BINS = [0, 25, 100, 250, 800, np.inf]
DIST_LABELS = ["0-25 NM", "25-100 NM", "100-250 NM", "250-800 NM", ">800 NM"]
s27_all = pd.concat([s27_tr, s27_te], ignore_index=True)
s27_all["distance_group"] = pd.cut(
    s27_all["Dist_To_Berth_NM"],
    bins=DIST_BINS,
    labels=DIST_LABELS,
    include_lowest=True,
)
dist_sum_27 = (
    s27_all.groupby(["split", "distance_group"], observed=True)["Target_Hours"]
    .agg(N="size", Mean="mean", Median="median", Max="max")
    .reset_index()
)
save_table(dist_sum_27, TASK03 / "sample_target_by_distance.csv")
display(dist_sum_27)

In [ ]:
for df_x in (s27_tr,):
    df_x["base_date_time"] = pd.to_datetime(
        df_x["base_date_time"], errors="coerce", utc=True
    )
    df_x["ata_time"] = pd.to_datetime(
        df_x["ata_time"], errors="coerce", utc=True
    )

tr_fs = s27_tr.loc[
    s27_tr["base_date_time"].lt(JUL) & s27_tr["ata_time"].lt(JUL)
].copy()
va_fs = s27_tr.loc[
    s27_tr["base_date_time"].ge(JUL)
    & s27_tr["base_date_time"].lt(OCT)
    & s27_tr["ata_time"].lt(OCT)
].copy()
ov_fs = set(tr_fs["trip_id"]) & set(va_fs["trip_id"])
tr_fs = tr_fs.loc[~tr_fs["trip_id"].isin(ov_fs)].copy()

features = [
    "sog", "latitude", "longitude", "length", "width", "draft",
    "Area_Proxy", "Volume_Proxy", "Is_Tanker",
    "cog_sin", "cog_cos", "heading_sin", "heading_cos",
]
if KNOWN_DESTINATION_AT_PREDICTION:
    features.append("Dist_To_Berth_NM")
features = [c for c in features if tr_fs[c].nunique(dropna=True) > 1]

if len(tr_fs) >= 20 and len(va_fs) >= 10 and len(features) >= 1:
    corr_mat = tr_fs[features].corr(min_periods=min(50, max(5, len(tr_fs))))
    corr_mat.to_csv(TASK03 / "train_pearson.csv", encoding="utf-8-sig")

    reduced = []
    drops = []
    for col in features:
        conflict = next(
            (
                k for k in reduced
                if pd.notna(corr_mat.loc[col, k])
                and abs(corr_mat.loc[col, k]) > 0.85
            ),
            None,
        )
        if conflict is None:
            reduced.append(col)
        else:
            drops.append({
                "removed": col,
                "retained": conflict,
                "r": float(corr_mat.loc[col, conflict]),
            })
    save_table(
        pd.DataFrame(drops, columns=["removed", "retained", "r"]),
        TASK03 / "proposed_feature_removals.csv",
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(features)))
    ax.set_xticklabels(features, rotation=90)
    ax.set_yticks(range(len(features)))
    ax.set_yticklabels(features)
    ax.set_title("Ma tran Pearson tren mau Train 27 cot")
    fig.colorbar(im, ax=ax)
    save_fig(fig, TASK03 / "02_pearson.png")


    def eval_metrics(y_true, y_pred) -> dict:
        """Compute regression and direction bias metrics."""
        err = np.asarray(y_pred) - np.asarray(y_true)
        return {
            "MAE_hours": float(np.mean(np.abs(err))),
            "RMSE_hours": float(np.sqrt(np.mean(err ** 2))),
            "bias_hours": float(np.mean(err)),
            "overprediction_pct": float(100.0 * np.mean(err > 0.0)),
        }


    def fit_lgb(cols: list[str], objective="regression"):
        """Train LightGBM probe purely on Jan-Jun / Jul-Sep."""
        model = lgb.LGBMRegressor(
            objective=objective,
            n_estimators=N_ESTIMATORS,
            learning_rate=0.05,
            num_leaves=31,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=N_JOBS,
            verbosity=-1,
            metric="l1",
        )
        model.fit(
            tr_fs[cols],
            tr_fs["Target_Hours"],
            eval_set=[(va_fs[cols], va_fs["Target_Hours"])],
            eval_metric="l1",
            callbacks=[
                lgb.early_stopping(
                    50, first_metric_only=True, verbose=False
                ),
                lgb.log_evaluation(0),
            ],
        )
        return model


    comp_rows = []
    grp_rows = []
    for name, cols in [
        ("domain_features", features),
        ("pearson_reduced", reduced),
    ]:
        mdl = fit_lgb(cols)
        prd = mdl.predict(va_fs[cols])
        comp_rows.append({
            "model": name,
            "n_features": len(cols),
            "best_iteration": mdl.best_iteration_,
            **eval_metrics(va_fs["Target_Hours"], prd),
        })
        gain = mdl.booster_.feature_importance(importance_type="gain")
        imp_df = pd.DataFrame({
            "feature": cols,
            "gain": gain,
            "gain_pct": 100.0 * gain / max(float(gain.sum()), 1e-12),
        }).sort_values("gain", ascending=False)
        save_table(imp_df, TASK03 / f"gain_{name}.csv")
        mdl.booster_.save_model(str(TASK03 / f"model_{name}.txt"))
        v_grps = pd.cut(
            va_fs["Dist_To_Berth_NM"],
            DIST_BINS,
            labels=DIST_LABELS,
            include_lowest=True,
        )
        for grp_lbl in DIST_LABELS:
            tk = (v_grps == grp_lbl).to_numpy()
            if tk.any():
                grp_rows.append({
                    "model": name,
                    "distance_group": grp_lbl,
                    "N": int(tk.sum()),
                    **eval_metrics(
                        va_fs["Target_Hours"].to_numpy()[tk], prd[tk]
                    ),
                })

    df_comp = pd.DataFrame(comp_rows)
    save_table(df_comp, TASK03 / "feature_selection_validation.csv")
    save_table(pd.DataFrame(grp_rows), TASK03 / "validation_by_distance.csv")
    display(df_comp)

    imp_plot = pd.read_csv(
        TASK03 / "gain_pearson_reduced.csv"
    ).sort_values("gain_pct")
    if len(imp_plot) > 0:
        fig, ax = plt.subplots(figsize=(9, 5))
        imp_plot.set_index("feature")["gain_pct"].plot.barh(
            ax=ax, color="#2878B5"
        )
        ax.set_xlabel("Gain (%)")
        ax.set_title("LightGBM Feature Importance (Ban 27 cot)")
        save_fig(fig, TASK03 / "03_gain.png")

    if RUN_ASYMMETRIC_EXPERIMENT:
        asym_rows = []


        def make_early_obj(alpha_val: float):
            """Return asymmetric L2 objective penalizing early preds."""
            def _obj(y_t, y_p):
                err = y_p - y_t
                w = np.where(err < 0.0, 1.0 + alpha_val, 1.0)
                return 2.0 * w * err, 2.0 * w

            return _obj


        for alpha_v in (0.0, 0.5, 1.0, 1.5):
            mdl_a = fit_lgb(reduced, objective=make_early_obj(alpha_v))
            prd_a = mdl_a.predict(va_fs[reduced])
            err_a = prd_a - va_fs["Target_Hours"].to_numpy()
            asym_rows.append({
                "alpha": alpha_v,
                "early_weight": 1.0 + alpha_v,
                "best_iteration": mdl_a.best_iteration_,
                **eval_metrics(va_fs["Target_Hours"], prd_a),
                "underprediction_pct": float(100.0 * np.mean(err_a < 0.0)),
                "early_cost_weight_2_5": float(
                    np.mean(
                        np.where(err_a < 0.0, 2.5, 1.0) * (err_a ** 2)
                    )
                ),
            })
        df_asym = pd.DataFrame(asym_rows)
        save_table(df_asym, TASK03 / "asymmetric_loss_validation.csv")
        display(df_asym)
else:
    print(
        f"Bo qua B2/B3 LightGBM probe do mau qua nho: "
        f"tr={len(tr_fs)}, va={len(va_fs)}, n_feat={len(features)}"
    )

In [ ]:
manifest = {
    "split_root": str(SPLIT_ROOT),
    "cutoff_date": SPLIT_CUTOFF_STR,
    "max_dist_tanker_nm": MAX_DIST_TANKER_NM,
    "max_dist_cargo_nm": MAX_DIST_CARGO_NM,
    "known_destination_at_prediction": KNOWN_DESTINATION_AT_PREDICTION,
    "raw18_scanned": {
        "train": raw18_tr["scanned"],
        "test": raw18_te["scanned"],
    },
    "base22_scanned": {"train": n_tr_22, "test": n_te_22, "all": n_all_22},
    "prep27_scanned": {"train": n27_tr, "test": n27_te},
    "overlap_trips": audit22["overlap_trips"],
    "days_2025_covered": audit22["days_covered"],
    "corr_base22_all": audit22["corr_all"],
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "lightgbm": lgb.__version__,
    },
}
(OUT / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

zip_path = OUT.parent / "bt2_dama_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob("*")):
        if (
            path.is_file()
            and path.suffix != ".parquet"
            and "duckdb_tmp" not in path.parts
        ):
            zf.write(path, path.relative_to(OUT))

print("Da luu toan bo ket qua BT2 tai:", OUT)
print("File ZIP tai ve 1-click:", zip_path)
display(FileLink(str(zip_path)))